In [ ]:
import traceback
from IPython.display import display, HTML

def show(html):
    display(HTML(html))

def box(msg, bg, border):
    show('<div style="font-family:sans-serif;padding:14px 18px;background:' + bg +
         ';border-left:4px solid ' + border + ';border-radius:6px;margin:8px 0;">'
         + msg + '</div>')

try:
    import subprocess
    import time
    import os
    import socket

    port  = 3838
    path  = '/home/jovyan/tools/wordfinder'
    name  = 'WordFinder'

    box('&#x23F3; Starting ' + name + '... please wait (up to 90 seconds).', '#f4f0f8', '#51247a')

    # Kill stale process on port (safe — ignore if fuser not available)
    try:
        subprocess.run(['fuser', '-k', str(port) + '/tcp'],
                       capture_output=True, check=False, timeout=5)
        time.sleep(1)
    except Exception:
        pass

    # Verify app directory exists before trying to start
    if not os.path.isdir(path):
        box('&#x274C; App directory not found: ' + path +
            '<br>Check that SLCLADAL/tools was cloned correctly by nbgitpuller.',
            '#fff0f0', '#e74c3c')
        raise SystemExit(0)

    app_r = os.path.join(path, 'app.R')
    if not os.path.isfile(app_r):
        box('&#x274C; app.R not found at: ' + app_r, '#fff0f0', '#e74c3c')
        raise SystemExit(0)

    box('&#x2139;&#xFE0F; Found app.R at ' + app_r, '#f0f7ff', '#4085C6')

    # Write R startup script to temp file
    r_script = '/tmp/start_wordfinder.R'
    with open(r_script, 'w') as f:
        f.write("shiny::runApp('" + path + "', port=" + str(port) +
                ", host='0.0.0.0', launch.browser=FALSE)\n")

    log_path = '/tmp/shiny_wordfinder.log'
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(
            ['Rscript', '--vanilla', r_script],
            stdout=log, stderr=subprocess.STDOUT
        )

    # Poll until port opens or process exits
    ready = False
    for i in range(120):
        time.sleep(1)
        if proc.poll() is not None:
            break
        try:
            with socket.create_connection(('localhost', port), timeout=1):
                pass
            ready = True
            break
        except OSError:
            pass

    # Read log
    try:
        with open(log_path) as f:
            log_txt = f.read()[-5000:]
    except Exception:
        log_txt = '(log not available)'

    if ready:
        base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
        tool_url = base_url.rstrip('/') + '/proxy/' + str(port) + '/'
        show(
            '<div style="font-family:sans-serif;padding:14px 18px;background:#eafaf1;'
            'border-left:4px solid #27ae60;border-radius:6px;margin-bottom:12px;">'
            '&#x2705; <b>' + name + ' is ready.</b> '
            '<a href="' + tool_url + '" target="_blank" '
            'style="margin-left:14px;padding:7px 20px;background:#51247a;'
            'color:white;border-radius:6px;text-decoration:none;font-weight:700;">'
            '&#x1F50D; Open ' + name + '</a></div>'
            '<iframe src="' + tool_url + '" width="100%" height="850" '
            'frameborder="0" style="border-radius:8px;border:1px solid #ddd;"></iframe>'
        )
    else:
        show(
            '<div style="font-family:sans-serif;padding:14px 18px;background:#fff0f0;'
            'border-left:4px solid #e74c3c;border-radius:6px;">'
            '<b style="color:#c0392b;">&#x274C; ' + name + ' failed to start.</b><br><br>'
            '<b>R log (last 5000 chars):</b><br>'
            '<pre style="background:#fff8f8;padding:10px;border-radius:4px;'
            'font-size:.78rem;white-space:pre-wrap;max-height:500px;overflow-y:auto;'
            'border:1px solid #fcc;">' + log_txt + '</pre></div>'
        )

except SystemExit:
    pass
except Exception as e:
    # Show full Python traceback directly on the Voila page
    tb = traceback.format_exc()
    show(
        '<div style="font-family:monospace;padding:14px 18px;background:#fff0f0;'
        'border-left:4px solid #e74c3c;border-radius:6px;">'
        '<b style="color:#c0392b;font-family:sans-serif;">'
        '&#x274C; Python error in launcher cell:</b><br><br>'
        '<pre style="background:#fff8f8;padding:10px;border-radius:4px;'
        'font-size:.8rem;white-space:pre-wrap;border:1px solid #fcc;">'
        + tb + '</pre></div>'
    )
